### Addressing class imbalance

In [ ]:
def train_two_phase(device, model, train_loader, val_loader, criterion, optimizer, num_epochs=15):
    """
    Phase 1: Aggressive cost-sensitive learning (focus on minority)
    Phase 2: Balanced fine-tuning (maintain overall accuracy)
    """

    phase1_epochs = int(num_epochs * 0.6)  # 60% of training 
    phase2_epochs = num_epochs - phase1_epochs 
 
    best_val_f1 = 0
    best_threshold = 0.5
    patience_counter = 0
    patience = 15
    
    print("="*80)
    print("PHASE 1: Aggressive Cost-Sensitive Learning")
    print("="*80)
    
    # Phase 1: High cost difference
    for epoch in tqdm(range(phase1_epochs),colour='GREEN'):  
        train_loss, train_time, threshold = train_loop_new(
            device, model, train_loader, criterion, optimizer, None,
            fn_cost=5.0,  # Very aggressive
            fp_cost=1.0,
            find_threshold=True,
            epoch=epoch,
            total_epochs=phase1_epochs
        )
        
        val_metrics = validate_with_threshold(model, val_loader, device, threshold, criterion)
        
        print(f"Epoch {epoch+1}/{phase1_epochs}: "
              f"Train_Loss = {train_loss:.4f}, "
              f"Validation_Loss = {val_metrics['loss']:.4f}, "
              f"Val_accuracy = {val_metrics['accuracy']:.4f}, "
              f"Val_balanced_accuracy = {val_metrics['balanced_accuracy']:.4f}, "
              f"F1 = {val_metrics['f1']:.4f}, "              
              f"Val_macro_auc = {val_metrics['macro_auc']:.4f}, "
              f"Val_macro_f1  = {val_metrics['macro_f1']:.4f}, " 
              f"Sensitivity = {val_metrics['sensitivity']:.4f}, "
              f"Specificity = {val_metrics['specificity']:.4f}, "              
              f"Threshold = {threshold:.3f}")
        
        if val_metrics['f1'] > best_val_f1: 
            best_val_f1 = val_metrics['f1']
            best_threshold = threshold
            torch.save(model.state_dict(), 'best_phase1.pth')
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping in Phase 1")
            break
    
    # Load best from phase 1
    model.load_state_dict(torch.load('best_phase1.pth'))
    
    print("\n" + "="*80)
    print("PHASE 2: Balanced Fine-Tuning")
    print("="*80)
    
    # Phase 2: Lower learning rate, reduced cost difference
    for param_group in optimizer.param_groups:
        param_group['lr'] *= 0.1  # Reduce learning rate
    
    patience_counter = 0
    
    for epoch in range(phase2_epochs):
        train_loss, train_time, threshold = train_loop_new(
            device, model, train_loader, criterion, optimizer, None,
            fn_cost=3.0,  # Much lower, more balanced
            fp_cost=1.0,
            find_threshold=True,
            epoch=epoch,
            total_epochs=phase2_epochs
        )
        
        val_metrics = validate_with_threshold(model, val_loader, device, threshold, criterion) 

        print(f"Epoch {epoch+1}/{phase1_epochs}: "
              f"Train_Loss = {train_loss:.4f}, "
              f"Validation_Loss = {val_metrics['loss']:.4f}, "
              f"Val_accuracy = {val_metrics['accuracy']:.4f}, "
              f"Val_balanced_accuracy = {val_metrics['balanced_accuracy']:.4f}, "
              f"F1 = {val_metrics['f1']:.4f}, "    
              f"Val_macro_auc = {val_metrics['macro_auc']:.4f}, "
              f"Val_macro_f1  = {val_metrics['macro_f1']:.4f}, " 
              f"Sensitivity = {val_metrics['sensitivity']:.4f}, "
              f"Specificity = {val_metrics['specificity']:.4f}, "  
              f"Threshold = {threshold:.3f}")
        
        if val_metrics['f1'] > best_val_f1:
            print("here")
            best_val_f1 = val_metrics['f1']
            best_threshold = threshold
            torch.save(model.state_dict(), 'best_final.pth')
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping in Phase 2")
            break

    if val_metrics['f1'] > best_val_f1:
        model.load_state_dict(torch.load('best_final.pth')) 
        print("here")
        
    return model, best_threshold

def validate_with_threshold(model, loader, device, threshold, criterion):
    """Validation with threshold and comprehensive metrics"""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    total_loss = 0
    
    with torch.no_grad():
        for data in loader:
            bag = data[0].to(device).float()
            label = data[1].to(device).long()  # Keep on device for loss calculation
            logits = model(bag)['logits']
            
            # Calculate loss
            loss = criterion(logits, label)
            total_loss += loss.item()
            
            # Get predictions with threshold
            probs = F.softmax(logits, 1)[:, 1]
            preds = (probs > threshold).cpu().numpy()
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds)
            all_labels.extend(label.cpu().numpy())
    
    from sklearn.metrics import (
        f1_score, recall_score, accuracy_score, balanced_accuracy_score,
        roc_auc_score, precision_score, confusion_matrix
    )
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    # Confusion matrix (binary case assumed: labels 0 and 1)
    cm = confusion_matrix(all_labels, all_preds)
    
    # Handle binary CM safely
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        # Fallback (e.g., if only one class is present) – you can also set to np.nan
        specificity = 0.0
    
    # Calculate all metrics
    metrics = {
        # Loss
        'loss': total_loss / len(loader),
        
        # Accuracy metrics
        'accuracy': accuracy_score(all_labels, all_preds),
        'balanced_accuracy': balanced_accuracy_score(all_labels, all_preds),
        
        # AUC metrics (using probabilities)
        'macro_auc': roc_auc_score(all_labels, all_probs, average='macro'),
        'micro_auc': roc_auc_score(all_labels, all_probs, average='micro'),
        'weighted_auc': roc_auc_score(all_labels, all_probs, average='weighted'),
        
        # F1 metrics
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'micro_f1': f1_score(all_labels, all_preds, average='micro', zero_division=0),
        'weighted_f1': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        
        # Precision metrics
        'macro_pre': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'micro_pre': precision_score(all_labels, all_preds, average='micro', zero_division=0),
        'weighted_pre': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        
        # Sensitivity (Recall)
        'sensitivity': recall_score(all_labels, all_preds, pos_label=1, zero_division=0),

        # Specificity (True Negative Rate)
        'specificity': specificity,
        
        # Confusion Matrix
        'confusion_matrix': confusion_matrix(all_labels, all_preds)
    }
    
    return metrics

def train_loop_new(device, model, loader, criterion, optimizer, scheduler, fn_cost=10.0, fp_cost=1.0, find_threshold=True, epoch=0, total_epochs=50):
    """Training loop with gradual cost annealing"""
    start = time.time()
    model.train()
    train_loss_log = 0
    all_probs, all_labels = [], []
    
    # Gradually reduce cost difference over epochs (annealing)
    # Start aggressive, end balanced
    progress = epoch / total_epochs
    current_fn_cost = fn_cost * (1.0 - 0.7 * progress) + 1.0 * (0.7 * progress)  # 10 -> 3
    current_fp_cost = fp_cost
    
    print(f"Epoch {epoch+1}: FN_cost={current_fn_cost:.2f}, FP_cost={current_fp_cost:.2f}")
    
    for i, data in enumerate(loader):
        optimizer.zero_grad()
        label = data[1].long().to(device)
        bag = data[0].to(device).float()
        train_logits = model(bag)['logits']
        
        # Cost-sensitive loss with current costs
        base_loss = criterion(train_logits, label)
        if base_loss.dim() > 0:
            preds = train_logits.argmax(1)
            costs = torch.ones_like(label, dtype=torch.float32)
            costs[(preds==0)&(label==1)] = current_fn_cost
            costs[(preds==1)&(label==0)] = current_fp_cost
            train_loss = (base_loss * costs).mean()
        else:
            train_loss = base_loss
        
        train_loss_log += train_loss.item()
        train_loss.backward()
        
        # Gradient clipping to prevent instability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        if find_threshold:
            with torch.no_grad():
                all_probs.extend(F.softmax(train_logits, 1)[:, 1].cpu().numpy())
                all_labels.extend(label.cpu().numpy())
    
    if scheduler is not None:
        scheduler.step()
    
    train_loss_log /= len(loader)
    
    # Find threshold
    threshold = 0.5
    if find_threshold and len(all_probs) > 0:
        best_f1 = 0
        for t in np.arange(0.2, 0.8, 0.05):  # Narrower range
            f1 = f1_score(all_labels, (np.array(all_probs) > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, threshold = f1, t
    
    end = time.time()
    
    return train_loss_log, end - start, threshold

In [ ]:
import numpy as np
from collections import Counter

def calculate_class_weights(train_loader):
    """Calculate class counts and weights from dataloader"""
    all_labels = []
    
    for data in train_loader:
        labels = data[1].numpy()
        all_labels.extend(labels)
    
    # Count samples per class
    class_counts = Counter(all_labels)
    num_classes = len(class_counts)
    
    print(f"Class distribution: {dict(class_counts)}")
    
    # Calculate weights (inverse frequency)
    total_samples = len(all_labels)
    class_weights = [total_samples / (num_classes * class_counts[i]) 
                     for i in range(num_classes)]
    
    print(f"Class weights: {class_weights}")
    
    # Return class_counts as list (ordered by class index)
    class_counts_list = [class_counts[i] for i in range(num_classes)]
    
    return class_counts_list, class_weights

def train_loop_class_imbalance(device, model, loader, optimizer, scheduler):
    start = time.time()
    model.train()
    train_loss_log = 0
    # Calculate weights from your data 
    class_counts, class_weights = calculate_class_weights(loader)
    
    criterion = FocalLossWithWeights(class_counts, gamma=2)

    for i, data in enumerate(loader):
        optimizer.zero_grad()
        label = data[1].long().to(device)
        bag = data[0].to(device).float()
        train_logits = model(bag)['logits']
        train_loss = criterion(train_logits, label)  
        train_loss_log += train_loss.item()
        train_loss.backward()
        optimizer.step()
    
    if scheduler is not None:
        scheduler.step()
    
    train_loss_log /= len(loader)
    end = time.time()
    total_time = end - start
    
    return train_loss_log, total_time

In [ ]:
# class weighted average with cross entropy and focal loss
# class FocalLossWithWeights(nn.Module):
#     """Focal Loss with automatic class weight calculation"""
    
#     def __init__(self, class_counts, gamma=2, reduction='mean'):
#         super(FocalLossWithWeights, self).__init__()
#         self.gamma = gamma
#         self.reduction = reduction
        
#         # Calculate inverse frequency weights
#         total = sum(class_counts)
#         self.alpha = torch.tensor([total / (len(class_counts) * c) for c in class_counts], 
#                                    dtype=torch.float32)
    
#     def forward(self, inputs, targets):
#         ce_loss = F.cross_entropy(inputs, targets, reduction='none')
#         pt = torch.exp(-ce_loss)
        
#         # Move alpha to the same device as inputs
#         alpha_t = self.alpha.to(inputs.device)[targets]  # ✓ FIXED
#         focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        
#         if self.reduction == 'mean':
#             return focal_loss.mean()
#         elif self.reduction == 'sum':
#             return focal_loss.sum()
#         else:
#             return focal_loss

In [ ]:
    # optimizer,base_lr = get_optimizer(args,mil_model)
    optimizer = torch.optim.AdamW(
    mil_model.parameters(), 
    lr=1e-4, 
    weight_decay=1e-3  # Increase from 1e-5
    )
    base_lr = 0.00001
model, best_threshold = train_two_phase(device, mil_model, train_dataloader, val_dataloader, criterion, optimizer, num_epochs=args.General.num_epochs)

In [6]:
import os

def list_files_in_directory(path):
    try:
        # Get all entries in the directory
        entries = os.listdir(path)
        
        # Filter out directories, keeping only files
        files = [f for f in entries if os.path.isfile(os.path.join(path, f))]
        
        return files
    except FileNotFoundError:
        return "The specified path does not exist."
    except PermissionError:
        return "Permission denied for this path."

# Example usage:
# Replace '/content/sample_data' with your actual folder path
folder_path = '/data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA' 
file_names = list_files_in_directory(folder_path)

print("Files found:", file_names)

Files found: ['TCGA-CF-A3MF-01Z-00-DX1.91768C26-5BB1-4DE6-9737-AA26BD687A75_BLCA.jpg', 'TCGA-E7-A7PW-01Z-00-DX1.27B92FC6-D50C-4816-A2C2-35E73D379DDD_BLCA.jpg', 'TCGA-ZF-AA52-01Z-00-DX1.BABC9074-8AF5-42E7-B3F8-39590121B620_BLCA.jpg', 'TCGA-E7-A97P-01Z-00-DX1.09FD969F-62C2-4C79-A826-657A7991C87B_BLCA.jpg', 'TCGA-XF-A9SI-01Z-00-DX1.537F0394-40FE-4CB7-8524-7BC34F624C15_BLCA.tiff', 'TCGA-ZF-A9RL-01Z-00-DX1.6A6B23B8-78D2-4468-8B25-A2D94130722C_BLCA.tiff']


In [7]:
import os

def check_files_ignore_ext(folder_path, input_list):
    if not os.path.exists(folder_path):
        return "Folder not found."

    # 1. Strip extensions from your input list just in case they are there
    clean_input = {os.path.splitext(f)[0] for f in input_list}

    # 2. Map clean names to their full filenames in the directory
    # This stores { "clean_name": "full_name_with_ext.svs" }
    dir_files_map = {os.path.splitext(f)[0]: f for f in os.listdir(folder_path)}

    found_full_names = []
    missing_names = []

    for name in clean_input:
        if name in dir_files_map:
            found_full_names.append(dir_files_map[name])
        else:
            missing_names.append(name)

    # Summary
    print("-" * 30)
    print(f"FILES FOUND:   {len(found_full_names)}")
    print(f"FILES MISSING: {len(missing_names)}")
    print("-" * 30)

    if missing_names:
        print("Missing (ID only):", missing_names)
        
    return found_full_names

# Your Input (with or without extensions)
tcga_list = [
    "TCGA-85-7696-01Z-00-DX1.d8756b4c-819f-4a5c-b148-125b8c6b3c27",
    "TCGA-85-7710-01Z-00-DX1.8bcfe326-1c19-4ebe-879d-83e77739f006",
    "TCGA-85-7844-01Z-00-DX1.9035209b-75ae-4c9b-8edf-5851af490f29",
    "TCGA-85-8048-01Z-00-DX1.1a663caa-dff1-4f03-b06b-c8b8b5ce08c6",
    "TCGA-85-8049-01Z-00-DX1.59554c0f-d04a-41f2-b152-654a848b0443",
    "TCGA-85-8052-01Z-00-DX1.26b66ae7-b73f-4263-85f7-e82dab5b657b",
    "TCGA-85-8071-01Z-00-DX1.876f6b0b-f615-43b2-8923-ee21f94568b7",
    "TCGA-85-8276-01Z-00-DX1.6306624d-e197-4744-a872-0a20413bd9fa",
    "TCGA-85-8277-01Z-00-DX1.ffad077e-e969-40ba-8504-2f2103911758",
    "TCGA-85-8288-01Z-00-DX1.0e68535f-a1f3-450c-b8fa-a859f2269da8"
] 

# Set your folder path here
target_dir = "/data_64T_3/Dataset/TCGA_LUSC/images/Tumor_DX1" 

found, lost = check_tcga_files(target_dir, tcga_list)

FILE CHECK SUMMARY
Total Expected: 10
Total Found:    0
Total Missing:  10

Missing Files:
  - TCGA-85-7696-01Z-00-DX1.d8756b4c-819f-4a5c-b148-125b8c6b3c27
  - TCGA-85-7710-01Z-00-DX1.8bcfe326-1c19-4ebe-879d-83e77739f006
  - TCGA-85-7844-01Z-00-DX1.9035209b-75ae-4c9b-8edf-5851af490f29
  - TCGA-85-8048-01Z-00-DX1.1a663caa-dff1-4f03-b06b-c8b8b5ce08c6
  - TCGA-85-8049-01Z-00-DX1.59554c0f-d04a-41f2-b152-654a848b0443
  - TCGA-85-8052-01Z-00-DX1.26b66ae7-b73f-4263-85f7-e82dab5b657b
  - TCGA-85-8071-01Z-00-DX1.876f6b0b-f615-43b2-8923-ee21f94568b7
  - TCGA-85-8276-01Z-00-DX1.6306624d-e197-4744-a872-0a20413bd9fa
  - TCGA-85-8277-01Z-00-DX1.ffad077e-e969-40ba-8504-2f2103911758
  - TCGA-85-8288-01Z-00-DX1.0e68535f-a1f3-450c-b8fa-a859f2269da8
